# Multi-Provider API Integration - Never Depend on One AI

**Module 7 · Lesson 7.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Three SDKs Side by Side - Same Task, Three APIs
- Unified Wrapper - One Interface for All Providers
- Fallback Routing - Automatic Failover
- Smart Routing - Best Model for Each Task
- Cost Tracking - Know Where Every Dollar Goes
- Project: Production Multi-Provider System

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" openai google-generativeai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## Monday, 9:04 AM: The App That Had No Plan B

This script has run in production for six months. It answers support questions. Nobody has touched it since March.

**📝 `monday_9am.py`** - *Intro*

In [ ]:
# This code worked for six months. Then Tuesday happened.
# Minimal example - deliberately no error handling; this lesson exists to fix that.
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
answer = client.chat.completions.create(
    model="gpt-5.4",
    messages=[{"role": "user", "content": "Summarize today's support tickets."}],
)
print(answer.choices[0].message.content)

# Output: Monday, 9:04 AM -
#   "Today's tickets cluster around three themes: login failures..."
#
# Output: Tuesday, 9:04 AM, during a provider outage -
#   Traceback (most recent call last):
#     ...
#   openai.InternalServerError: Error code: 503 - upstream connect error
#
# Your dashboard: 4,000 users staring at a spinner.
# The status page: "Elevated error rates. Investigating."
# Your code: has no Plan B.

One import, one client, one point of failure. That `openai.InternalServerError` is a real exception class, and by Step 3 you will catch it, skip the dead provider, and let Claude answer instead - your users never see the spinner. First we build that safety net by hand so you understand every moving part; then, in Step 8, you'll see why most production teams buy it off the shelf.

> 📦 **Info**
>
> 🔑 Before you start: keys, env vars, and three words from Module 5
> - **API keys:** each provider issues keys from its console - OpenAI (platform.openai.com), Anthropic (console.anthropic.com), Google AI Studio (aistudio.google.com). Free tiers or small credits are enough for this lesson.
> - **Store them as environment variables**, never in code: put `OPENAI_API_KEY=sk-...` lines in a `.env` file, load with `python-dotenv` (`pip install python-dotenv`, then `from dotenv import load_dotenv; load_dotenv()`). `os.getenv("NAME")` returns `None` silently if the variable is missing - the #1 first-run gotcha.
> - **Token:** the unit providers bill by (roughly 3/4 of an English word). **temperature:** randomness dial, 0 = focused, 1 = creative - we use 0.3 for support-style answers. **max_tokens:** output cap - Anthropic REQUIRES it, the others default it. All three were introduced in Module 5; that is all you need here.
> - **Run the lesson's files in one session** (one notebook or one REPL): each part builds on classes defined in the previous part.

Sixty-second warm-up - three cards from the lessons this one stands on. Tap each to flip it.

## Why You Need Multiple Providers

> 🖧 **Analogy**
>
> **Think of a hospital's power supply.** They don't rely on one power line. They have: main grid power, a backup generator, and a battery UPS. If the grid goes down, the generator kicks in within seconds. If the generator fails, batteries keep life-support running. Patients never notice the switch.
> Your AI app needs the same resilience. **One provider as primary, a second as fallback, a third as emergency backup.** If one goes down or gets slow, your code automatically switches to the next one. Users never notice.
> **Where this analogy breaks down:** backup power delivers IDENTICAL electricity - a fallback LLM delivers a DIFFERENT answer. Models drift in style, capability, and even which parameters they accept. And unlike a UPS, the switch itself is not free: every retry and backoff second is user-facing latency you pay for. Step 3's animation makes that cost visible.

> 📦 **Info**
>
> Real incidents that break single-provider apps (from the providers' own status pages)
> - **2024-12-11:** all OpenAI services degraded or unavailable for ~4.5 hours - a telemetry deployment overwhelmed the Kubernetes control plane ([status.openai.com/history](https://status.openai.com/history))
> - **2024-12-26:** >90% error rates on OpenAI APIs for ~9 hours - a data-center power failure (status.openai.com)
> - **2025-06-10:** a ~15-hour OpenAI API + ChatGPT outage; recovery was slow because emergency "break-glass" deploy tooling didn't exist yet (status.openai.com)
> - **Model lifecycle churn, 2025-26:** Google ended support for its dominant Python SDK (google-generativeai, 2025-11-30); Anthropic retired the Claude Sonnet 4 snapshot thousands of tutorials hardcoded (2026-06-15); DeepSeek deprecated its `deepseek-chat` alias on ~3 weeks' notice (2026-07-24)
> Every one of these breaks a single-provider app with zero changes to your code. All survivable with a multi-provider strategy.

> ℹ️ **Info**
>
> What you'll build
> - **Three provider SDKs** - Gemini, OpenAI, and Anthropic side by side
> - **Unified wrapper class** - one interface that works with any provider
> - **Fallback routing** - automatic failover when a provider goes down
> - **Smart routing** - send each task to the best provider for that task type
> - **Cost tracking** - monitor spend per provider in real-time
> - **Project** - production-grade multi-provider system with health checks

---

## 🎯 Concept 1: Three SDKs Side by Side - Same Task, Three APIs

### Three SDKs Side by Side - Same Task, Three APIs

See how the same question is asked to Gemini, OpenAI, and Anthropic.

> 💬 **Analogy**
>
> **Three phone networks, same call.** Jio, Airtel, and Vi all let you make calls. Each has a slightly different dialing interface, but the result is the same - you talk to someone. Similarly, Gemini, OpenAI, and Anthropic all answer questions. Each has a different SDK, but the concept is identical: send a message, get a response.

**📝 `part1a_gemini.py`** - *Gemini*

In [ ]:
# =============================================
# PROVIDER 1: Google Gemini
# pip install google-generativeai
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

response = model.generate_content("Explain API rate limits in 2 sentences.")
print(f"Gemini: {response.text}")

# With system prompt
model = genai.GenerativeModel("gemini-2.0-flash",
    system_instruction="You are a helpful AI tutor.")
chat = model.start_chat()
response = chat.send_message("Why do APIs have rate limits?")
print(f"Gemini (chat): {response.text}")

# Output: (representative run)
# Gemini: A rate limit caps how many requests a client may send per time
# window. Providers use them to protect shared capacity and to bill fairly.
# Gemini (chat): APIs enforce rate limits to keep one heavy user from...
# NOTE: google-generativeai reached end-of-life 2025-11-30 - migrating this
# lesson to the google-genai SDK is tracked in plan-7.1.md (retrofit PR 2).

**📝 `part1b_openai.py`** - *OpenAI*

In [ ]:
# =============================================
# PROVIDER 2: OpenAI (GPT-4o)
# Skipping error handling in the side-by-side demos - see production version in Part 3
# pip install openai
# =============================================

import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful AI tutor."},
        {"role": "user", "content": "Explain API rate limits in 2 sentences."},
    ],
    temperature=0.3,
)

print(f"OpenAI: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")

# Output: (representative run)
# OpenAI: Rate limits cap how many requests or tokens you can send per
# minute. They protect the provider's infrastructure and your bill.
# Tokens: 87

**📝 `part1c_anthropic.py`** - *Anthropic*

In [ ]:
# =============================================
# PROVIDER 3: Anthropic (Claude)
# Skipping error handling in the side-by-side demos - see production version in Part 3
# pip install anthropic
# =============================================

import os
from anthropic import Anthropic

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="You are a helpful AI tutor.",
    messages=[
        {"role": "user", "content": "Explain API rate limits in 2 sentences."},
    ],
)

print(f"Claude: {response.content[0].text}")
print(f"Tokens: {response.usage.input_tokens + response.usage.output_tokens}")

# Output: (representative run)
# Claude: A rate limit is a ceiling on requests or tokens per time window,
# enforced so shared capacity stays fair and predictable for every client.
# Tokens: 102
# Note the asymmetry: Anthropic REQUIRES max_tokens; OpenAI and Gemini default it.

#### 💡 What just happened

What just happened?
  Same question ("Explain API rate limits in 2 sentences"), three different SDKs: Gemini uses `generate_content()`, OpenAI uses `chat.completions.create()`, Anthropic uses `messages.create()`. Each has slightly different parameter names (system_instruction vs system vs system), response structures (response.text vs response.choices[0].message.content vs response.content[0].text), and token counting. **The differences are cosmetic. The concept is identical.** Now let's unify them.

---

## 🎯 Concept 2: Unified Wrapper - One Interface for All Providers

### Unified Wrapper - One Interface for All Providers

Write code ONCE. Swap providers by changing one string. No code changes needed.

> 🔌 **Analogy**
>
> **Think of a universal power adapter.** You travel to the UK (3-pin plugs), Europe (round pins), and India (mixed). Instead of carrying 3 different chargers, you carry ONE universal adapter that converts any plug to any socket. Your laptop doesn't know or care which country's power it's using. Our unified wrapper does the same for AI: your app code doesn't know which provider is running underneath.

**📝 `part2_unified_wrapper.py Python`** - *Class*

In [ ]:
# =============================================
# UNIFIED LLM WRAPPER
# One interface. Three providers.
# Swap providers by changing one string.
# =============================================

import os, time
from dataclasses import dataclass

@dataclass
class LLMResponse:
    """Unified response format - same regardless of provider."""
    text: str
    provider: str
    model: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    cost_usd: float

class UnifiedLLM:
    """Call any LLM provider through one consistent interface."""
    
    # Pricing per 1M tokens (approximate, 2025-2026)
    PRICING = {
        "gemini-2.0-flash":          {"input": 0.075, "output": 0.30},
        "gemini-2.5-pro":            {"input": 1.25,  "output": 10.00},
        "gpt-4o":                    {"input": 2.50,  "output": 10.00},
        "gpt-4o-mini":               {"input": 0.15,  "output": 0.60},
        "claude-sonnet-4-6":  {"input": 3.00,  "output": 15.00},
        "claude-haiku-4-5":          {"input": 1.00,  "output": 5.00},
    }
    
    def __init__(self):
        self._clients = {}
        self._init_providers()
    
    def _init_providers(self):
        # Initialize available providers (only if API key exists)
        if os.getenv("GOOGLE_API_KEY"):
            import google.generativeai as genai
            genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
            self._clients["gemini"] = genai
        
        if os.getenv("OPENAI_API_KEY"):
            from openai import OpenAI
            self._clients["openai"] = OpenAI()
        
        if os.getenv("ANTHROPIC_API_KEY"):
            from anthropic import Anthropic
            self._clients["anthropic"] = Anthropic()
        
        print(f"Initialized providers: {list(self._clients.keys())}")
    
    def _calc_cost(self, model, input_tok, output_tok):
        pricing = self.PRICING.get(model, {"input": 0, "output": 0})
        return (input_tok * pricing["input"] + output_tok * pricing["output"]) / 1_000_000
    
    def _detect_provider(self, model):
        if "gemini" in model: return "gemini"
        if "gpt" in model: return "openai"
        if "claude" in model: return "anthropic"
        raise ValueError(f"Unknown model: {model}")
    
    def chat(self, message: str, model: str = "gemini-2.0-flash",
            system: str = "", temperature: float = 0.3) -> LLMResponse:
        """Send a message to any provider. Returns unified response."""
        provider = self._detect_provider(model)
        start = time.time()
        
        if provider == "gemini":
            m = self._clients["gemini"].GenerativeModel(model,
                system_instruction=system or None,
                generation_config={"temperature": temperature})
            resp = m.generate_content(message)
            text = resp.text
            in_tok = resp.usage_metadata.prompt_token_count
            out_tok = resp.usage_metadata.candidates_token_count
        
        elif provider == "openai":
            msgs = []
            if system: msgs.append({"role": "system", "content": system})
            msgs.append({"role": "user", "content": message})
            resp = self._clients["openai"].chat.completions.create(
                model=model, messages=msgs, temperature=temperature)
            text = resp.choices[0].message.content
            in_tok = resp.usage.prompt_tokens
            out_tok = resp.usage.completion_tokens
        
        elif provider == "anthropic":
            resp = self._clients["anthropic"].messages.create(
                model=model, max_tokens=2048,
                system=system or "You are a helpful assistant.",
                messages=[{"role": "user", "content": message}],
                temperature=temperature)
            text = resp.content[0].text
            in_tok = resp.usage.input_tokens
            out_tok = resp.usage.output_tokens
        
        latency = (time.time() - start) * 1000
        cost = self._calc_cost(model, in_tok, out_tok)
        
        return LLMResponse(text=text, provider=provider, model=model,
                          input_tokens=in_tok, output_tokens=out_tok,
                          latency_ms=latency, cost_usd=cost)

# ── Use it! Same code, any provider ──
llm = UnifiedLLM()

# Switch providers by changing one string:
for model in ["gemini-2.0-flash", "gpt-4o-mini", "claude-haiku-4-5"]:
    resp = llm.chat("What is RAG in one sentence?", model=model)
    print(f"  [{resp.provider:9s}] {resp.latency_ms:6.0f}ms  ${resp.cost_usd:.5f}  {resp.text[:80]}")

# Output: (representative run)
# Initialized providers: ['gemini', 'openai', 'anthropic']
#   [gemini   ]    412ms  $0.00004  A rate limit caps how many requests a client...
#   [openai   ]    633ms  $0.00009  Rate limits protect shared infrastructure and...
#   [anthropic]    587ms  $0.00013  A rate limit is a ceiling on requests or tok...

- `LLMResponse` - the universal power adapter's OUTPUT socket: whatever provider answered, your app always receives the same 7 fields (text, provider, model, tokens in/out, latency, cost).

- `_init_providers` - walks the three API keys and only plugs in providers whose key exists. No key, no client - the wrapper degrades gracefully instead of crashing at import.

- `_detect_provider` - a substring sniff ("gemini" / "gpt" / "claude") that picks which adapter plug to use. Cheap but brittle: an ID like `gpt-oss-120b` served elsewhere would fool it - production gateways use explicit registries.

- `_calc_cost` - multiplies each direction's tokens by its per-1M price. Unknown model = $0, which silently blinds the budget guard - a real gateway fails loudly here.

- `chat()` - the adapter itself: three branches translate one call into each provider's dialect, then normalize the response back into `LLMResponse`.

Read it as: *one socket in, three plugs out, one socket back.*

#### 💡 What just happened

What just happened?
  We built a `UnifiedLLM` class with ONE `.chat()` method that works with any provider. Change the model string from "gemini-2.0-flash" to "gpt-4o" to "claude-sonnet-4-6" - same code, same response format, same cost tracking. The `LLMResponse` dataclass normalizes everything: text, provider name, token counts, latency, and cost. **Your app never touches provider-specific code.**

---

## 🎯 Concept 3: Fallback Routing - Automatic Failover

### Fallback Routing - Automatic Failover

If Provider A fails, automatically try Provider B. If B fails, try C. Zero downtime.

> 💡 **Info**
>
> ⚠️ The naive approach - and why it breaks
> A common mistake is wrapping every call in a bare `except Exception` and retrying everything with backoff. This would fail you exactly when clarity matters: a typo'd API key (a 401) gets retried and slept on instead of failing fast, burning seconds while hiding the real problem. Retry only what is retryable - rate limits (429) and server errors (5xx); fail fast on authentication and bad-request errors. The router below still uses the broad catch for teaching brevity; Exercise 4 upgrades it to typed exceptions and a circuit breaker.

**📝 `part3_fallback_router.py Python`** - *Class*

In [ ]:
# =============================================
# FALLBACK ROUTER: Automatic failover + retry
# Try primary → fallback 1 → fallback 2
# with exponential backoff retry per provider.
# =============================================

import time, logging
from collections import defaultdict

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("llm_router")

class FallbackRouter:
    """Route requests with automatic fallback and retry."""
    
    def __init__(self, llm: UnifiedLLM,
                 chain: list[str] = None,
                 max_retries: int = 2,
                 timeout_ms: float = 15000):
        self.llm = llm
        self.chain = chain or ["gemini-2.0-flash", "gpt-4o-mini", "claude-haiku-4-5"]
        self.max_retries = max_retries
        self.timeout_ms = timeout_ms
        # defaultdict, not a fixed dict: SmartRouter swaps the chain later,
        # and stats must not KeyError on models it has never seen
        self.stats = defaultdict(lambda: {"success": 0, "fail": 0})
    
    def _try_with_retry(self, message, model, system, temperature):
        """Try a model with exponential backoff retry."""
        for attempt in range(self.max_retries):
            try:
                resp = self.llm.chat(message, model=model,
                                    system=system, temperature=temperature)
                
                # Check latency threshold
                if resp.latency_ms > self.timeout_ms:
                    log.warning(f"{model}: too slow ({resp.latency_ms:.0f}ms)")
                    raise TimeoutError("Latency exceeded threshold")
                
                self.stats[model]["success"] += 1
                return resp
            
            except Exception as e:
                log.warning(f"{model} attempt {attempt+1} failed: {e}")
                if attempt < self.max_retries - 1:  # never sleep after the LAST attempt -
                    wait = 2 ** attempt             # that only delays the failover
                    log.warning(f"Retry in {wait}s")
                    time.sleep(wait)
        
        self.stats[model]["fail"] += 1
        return None
    
    def route(self, message: str, system: str = "",
             temperature: float = 0.3) -> LLMResponse:
        """Try each model in the chain until one succeeds."""
        
        for model in self.chain:
            log.info(f"Trying: {model}")
            resp = self._try_with_retry(message, model, system, temperature)
            
            if resp:
                if model != self.chain[0]:
                    log.info(f"Used fallback: {model}")
                return resp
        
        raise RuntimeError("All providers failed!")
    
    def health_report(self):
        print("\nProvider Health Report:")
        for model, stats in self.stats.items():
            total = stats["success"] + stats["fail"]
            rate = stats["success"] / total * 100 if total > 0 else 0
            status = "🟢" if rate > 90 else "🟡" if rate > 50 else "🔴"
            print(f"  {status} {model:30s}  {stats['success']}/{total} ({rate:.0f}%)")

# ── Use it ──
router = FallbackRouter(llm)

# This automatically falls back if the primary fails
response = router.route(
    "What are the benefits of vector databases for RAG?",
    system="You are a senior AI engineer. Be concise.",
)

print(f"\nProvider used: {response.provider}")
print(f"Latency: {response.latency_ms:.0f}ms")
print(f"Cost: ${response.cost_usd:.5f}")
print(f"Answer: {response.text[:150]}...")

router.health_report()

# Output: (representative run, primary forced down to show the failover)
# INFO:llm_router:Trying: gemini-2.0-flash
# WARNING:llm_router:gemini-2.0-flash attempt 1 failed: 503 Service Unavailable
# WARNING:llm_router:Retry in 1s
# WARNING:llm_router:gemini-2.0-flash attempt 2 failed: 503 Service Unavailable
# INFO:llm_router:Trying: gpt-4o-mini
# INFO:llm_router:Used fallback: gpt-4o-mini
#
# Provider used: openai
# Latency: 641ms   Cost: $0.00011
# Answer: Retry a failed request when the error is transient - a 429 or 5xx...
#
# Provider Health Report:
#   🔴 gemini-2.0-flash    0/1 (0%)
#   🟢 gpt-4o-mini         1/1 (100%)

- `chain` - the hospital's power priority list: grid, generator, UPS. Order IS the policy.

- `_try_with_retry` - gives ONE provider its full chance: up to `max_retries` attempts with exponential backoff between them (1s, then 2s), and no sleep after the last attempt - sleeping there would only delay the switch to the generator.

- `stats` (a `defaultdict`) - the maintenance logbook: every success and failure per model, so `health_report()` can show which power source keeps browning out. A defaultdict, because Step 4 will swap new models into the chain at runtime.

- `resp.latency_ms > timeout_ms` - honesty check: this fires AFTER the response arrives, so the slow call is already paid for. A true timeout is the SDK's `timeout=` parameter, which cancels the request - that upgrade is exercise 4's job.

- `route()` - walks the chain until someone answers; only if grid, generator AND batteries are dead does it raise.

Read it as: *give each source a fair chance, log everything, move down the priority list.*

#### 💡 What just happened

What just happened?
  We built a `FallbackRouter` with: (1) **ordered chain** - try Gemini first, then OpenAI, then Claude. (2) **retry with exponential backoff** - if a request fails, wait 1s and retry; if it fails again, wait 2s and retry. (3) **latency threshold** - if a provider responds but is too slow (>15 seconds), treat it as failed and try the next one. (4) **health tracking** - counts successes and failures per provider so you can monitor reliability. **Users see an answer instead of an error - but failover is NOT free.** With max_retries=2, a dead primary still costs one 1-second backoff before the router moves on; a dead primary AND fallback cost ~2+ seconds before the last resort answers. That retry-tax is the tradeoff you tune: fewer retries = faster failover but more false alarms on transient blips. The animation below lets you watch exactly where those seconds go.

- **Scene 1 (happy path):** the request dot flies to the primary and comes straight back. Boring on purpose - this is your Monday. Do: click a provider's health badge to cycle it healthy → slow → down → rate-limited.

- **Scene 2 (primary down):** watch the backoff countdown actually tick - 1.0s, then 2.0s - before the dot moves to the fallback. Notice: the latency counter keeps running the whole time. That is your user's spinner.

- **Scene 3 (all down):** the error carries every provider's exception - the debugging detail the naive version throws away. Also try "slow": the response comes back, gets PAID for, and is discarded - the cost counter still increments.

Controls: Play auto-sends requests; Send request fires one; Reset clears; Speed = Slow / Normal / Fast; scene buttons or arrow keys switch scenarios.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

- **Scene 1 (trip + cooldown):** send requests at a dead provider and watch the 5-segment failure counter fill until the breaker TRIPS to OPEN. The cooldown then counts down in compressed time. Notice: the race panel - the naive lane burns ~3 seconds per request in retry sleeps; the breaker lane fails fast at ~0ms while OPEN.

- **Scene 2 (the probe):** after cooldown the breaker goes HALF_OPEN and lets exactly ONE probe through. Your health toggle decides: probe succeeds → CLOSED (recovered); probe fails → back to OPEN for another cooldown. Do: flip the provider back to healthy mid-cooldown and watch recovery happen on the next probe.

Controls: Play auto-sends; Send request fires one; Provider health toggles the backend; Reset clears; Speed = Slow / Normal / Fast.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Smart Routing - Best Model for Each Task

### Smart Routing - Best Model for Each Task

Different tasks need different strengths. Route each task to the best provider for it.

> 🛠 **Analogy**
>
> **Think of a hospital with specialists.** You don't send every patient to the same doctor. Broken bone → orthopedic surgeon. Heart issue → cardiologist. Skin rash → dermatologist. Each specialist is the best at their specific thing. Smart routing does the same: coding questions → GPT-4o (great at code). Long document analysis → Gemini (1M token window). Careful reasoning → Claude (strong at nuance).

**📝 `part4_smart_router.py Python`** - *Class*

In [ ]:
# =============================================
# SMART ROUTER: Send each task to the BEST model
# for that specific task type.
# =============================================

class SmartRouter:
    """Route tasks to the best provider based on task type."""
    
    # Which model is best at what?
    ROUTING_TABLE = {
        "code":         {"primary": "gpt-4o",            "fallback": "gemini-2.5-pro"},
        "long_doc":     {"primary": "gemini-2.0-flash",  "fallback": "claude-sonnet-4-6"},
        "reasoning":    {"primary": "claude-sonnet-4-6", "fallback": "gpt-4o"},
        "simple":       {"primary": "gemini-2.0-flash",  "fallback": "gpt-4o-mini"},
        "creative":     {"primary": "claude-sonnet-4-6", "fallback": "gpt-4o"},
        "structured":   {"primary": "gemini-2.0-flash",  "fallback": "gpt-4o-mini"},
    }
    
    def __init__(self, llm: UnifiedLLM):
        self.llm = llm
        self.fallback_router = FallbackRouter(llm)
    
    def classify_task(self, message: str) -> str:
        """Detect what kind of task this is (cheap heuristic)."""
        msg = message.lower()
        
        if any(w in msg for w in ["code", "function", "debug", "python", "javascript", "api", "error"]):
            return "code"
        if any(w in msg for w in ["summarize", "document", "analyze this", "read this"]):
            return "long_doc"
        if any(w in msg for w in ["think", "reason", "pros and cons", "compare", "ethics"]):
            return "reasoning"
        if any(w in msg for w in ["write", "story", "creative", "poem", "essay"]):
            return "creative"
        if any(w in msg for w in ["json", "extract", "classify", "format"]):
            return "structured"
        return "simple"
    
    def route(self, message: str, system: str = "", **kwargs) -> LLMResponse:
        """Automatically route to the best model for this task."""
        task = self.classify_task(message)
        routing = self.ROUTING_TABLE[task]
        
        chain = [routing["primary"], routing["fallback"]]
        self.fallback_router.chain = chain
        
        resp = self.fallback_router.route(message, system=system, **kwargs)
        print(f"  Task: {task:12s} → Model: {resp.model:30s} (${resp.cost_usd:.5f})")
        return resp

# ── Test with different task types ──
smart = SmartRouter(llm)

tasks = [
    "Write a Python function that sorts a list using merge sort.",
    "Summarize this 50-page document about climate change.",
    "Compare the pros and cons of microservices vs monolith architecture.",
    "Write a short story about a robot discovering music.",
    "Extract product name and price from this review as JSON.",
    "What is the capital of France?",
]

print("Smart routing results:\n")
for task in tasks:
    smart.route(task)

# Output: (representative run)
# Smart routing results:
#   Task: code         → Model: gpt-4o                    ($0.00310)
#   Task: long_doc     → Model: gemini-2.0-flash          ($0.00007)
#   Task: reasoning    → Model: claude-sonnet-4-6         ($0.00214)
#   Task: creative     → Model: claude-sonnet-4-6         ($0.00198)
#   Task: structured   → Model: gemini-2.0-flash          ($0.00006)
#   Task: simple       → Model: gemini-2.0-flash          ($0.00005)
# Note the 60x cost spread between the cheapest and priciest route -
# that spread is the whole business case for routing (animation below).

- `ROUTING_TABLE` - the hospital's triage board: which specialist (model) sees which kind of patient (task), each with a named backup. This table is OPINION, not fact - it goes stale as models leapfrog each other, and maintaining it is the hidden cost of routing.

- `classify_task` - the triage nurse, implemented as keyword matching. Fast and free, but literal: "I got an error ordering pizza" contains "error", so it gets sent to the expensive code specialist. The animation below has that exact card.

- `route()` - swaps the fallback router's chain to [specialist, backup] for this one request, then lets Step 3's machinery do the resilience work. Note the smell: mutating shared state per call works in a notebook, but two concurrent requests would race on `self.fallback_router.chain` - a real gateway passes the chain per-request.

Read it as: *triage first, then the normal failover machinery - routing rides ON TOP of resilience, it does not replace it.*

#### 💡 What just happened

What just happened?
  We built a `SmartRouter` that classifies each request by task type (code, document, reasoning, creative, structured, simple) and routes it to the best model. Coding questions go to GPT-4o. Long document analysis goes to Gemini (1M token context). Nuanced reasoning goes to Claude. Simple questions go to the cheapest model. Each route has a fallback in case the primary is down. **You get the best quality AND lowest cost for every request type.** This pattern has real research behind it: LLM cascades were formalized in [FrugalGPT (Chen et al., 2023)](https://arxiv.org/abs/2305.05176) and learned routers in [RouteLLM (Ong et al., 2024)](https://arxiv.org/abs/2406.18665), which preserved ~95% of flagship quality at a fraction of the cost.

- **Scene 1 (route the traffic):** task cards deal through the keyword classifier into three model lanes; each card shows its real cost (computed from the same per-1M prices as the lesson's PRICING table). Notice: the dual ledger - "everything on the flagship" vs "routed" - and the monthly projection at 100K requests diverging in real time.

- **Scene 2 (the misclassification tax):** the deck turns ambiguous - "I got an error ordering pizza" contains the word "error", so the keyword classifier ships it to the expensive code lane (it flashes rose). Watch the savings % erode. Notice: routing is a bet on a classifier and a price table, and both need maintenance.

Controls: Play auto-deals; Deal card deals one; Reset clears the ledgers; Speed = Slow / Normal / Fast; scene buttons or arrow keys.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Cost Tracking - Know Where Every Dollar Goes

### Cost Tracking - Know Where Every Dollar Goes

Monitor spend per provider, per task, per hour. Catch cost spikes before they hit your wallet.

**📝 `part5_cost_tracker.py Python`** - *Class*

In [ ]:
# =============================================
# COST TRACKER: Real-time spend monitoring
# =============================================

from collections import defaultdict
from datetime import datetime

class CostTracker:
    """Track LLM spending per provider, model, and time period."""
    
    def __init__(self, daily_budget_usd: float = 10.0):
        self.daily_budget = daily_budget_usd
        self.records = []
    
    def log(self, response: LLMResponse):
        self.records.append({
            "timestamp": datetime.now(),
            "provider": response.provider,
            "model": response.model,
            "input_tokens": response.input_tokens,
            "output_tokens": response.output_tokens,
            "cost": response.cost_usd,
            "latency_ms": response.latency_ms,
        })
        
        # Budget alert
        today_cost = self.today_spend()
        if today_cost > self.daily_budget * 0.8:
            print(f"  ⚠️ BUDGET WARNING: ${today_cost:.2f} / ${self.daily_budget:.2f} today")
    
    def today_spend(self):
        today = datetime.now().date()
        return sum(r["cost"] for r in self.records if r["timestamp"].date() == today)
    
    def report(self):
        print("\n💰 Cost Report")
        print("─" * 50)
        
        # Per provider
        by_provider = defaultdict(lambda: {"cost": 0, "calls": 0, "tokens": 0})
        for r in self.records:
            p = by_provider[r["provider"]]
            p["cost"] += r["cost"]
            p["calls"] += 1
            p["tokens"] += r["input_tokens"] + r["output_tokens"]
        
        print(f"\n{'Provider':<12} {'Calls':>6} {'Tokens':>10} {'Cost':>10} {'Avg/call':>10}")
        total_cost = 0
        for prov, data in sorted(by_provider.items()):
            avg = data["cost"] / data["calls"] if data["calls"] else 0
            print(f"  {prov:<12} {data['calls']:>4} {data['tokens']:>8,} ${data['cost']:>8.4f} ${avg:>8.5f}")
            total_cost += data["cost"]
        
        print(f"\n  Total: ${total_cost:.4f}  |  Today: ${self.today_spend():.4f} / ${self.daily_budget:.2f}")

# ── Wire it up with the router ──
tracker = CostTracker(daily_budget_usd=5.0)

for task in tasks:
    resp = smart.route(task)
    tracker.log(resp)

tracker.report()

# Output: (representative run)
# 💰 Cost Report
# ──────────────────────────────────────────────────
# Provider      Calls     Tokens       Cost   Avg/call
#   anthropic       2      1,842    $0.0041   $0.00206
#   gemini          3      1,201    $0.0002   $0.00006
#   openai          1        698    $0.0031   $0.00310
#
#   Total: $0.0074  |  Today: $0.0074 / $5.00

- `log()` - the till receipt: one record per call with timestamp, provider, model, both token counts, cost, latency. Everything downstream (budgets, reports) is arithmetic over this list.

- `today_spend()` - filters receipts to today's date using the local wall clock. Fine for a demo; a production meter would use UTC and persistent storage (this list lives in memory and vanishes on restart).

- the 80% check inside `log()` - the low-fuel light: it warns BEFORE the budget is gone, because a warning at 100% is a post-mortem, not an alert.

- `report()` - groups receipts by provider and prints calls / tokens / cost / average - the table you paste into the design review when someone asks "what does multi-provider actually cost us?"

Read it as: *meter every call at the moment it happens; reports are just queries over the receipts.*

#### 💡 What just happened

What just happened?
  We built a `CostTracker` that logs every API call with timestamps, tokens, cost, and latency. It provides: per-provider cost breakdown, daily spend tracking, and budget alerts (warning at 80% of daily limit). The report shows calls, total tokens, total cost, and average cost per call per provider. **This is how production AI companies keep costs predictable** - you know exactly how much each provider costs per call and can optimize routing to minimize spend. Caching and batch discounts multiply these savings - details in Lesson 7.3 (Cost Engineering).

---

## 🎯 Concept 6: Project: Production Multi-Provider System

### Project: Production Multi-Provider System

Everything combined: unified wrapper + smart routing + fallback + cost tracking + health checks.

**📝 `project_production_system.py Complete`** - *Project*

In [ ]:
# =============================================
# PRODUCTION MULTI-PROVIDER SYSTEM
# All components wired together.
# Drop this into any Python app.
# =============================================

class ProductionLLM:
    """Production-grade LLM client with multi-provider support."""
    
    def __init__(self, daily_budget: float = 10.0):
        self.llm = UnifiedLLM()
        self.router = SmartRouter(self.llm)
        self.tracker = CostTracker(daily_budget_usd=daily_budget)
    
    def ask(self, message: str, system: str = "",
           model: str = None, temperature: float = 0.3) -> LLMResponse:
        """
        The ONE method you use in your app.
        
        - model=None → auto-route to best model for the task
        - model="gpt-4o" → use specific model with fallback
        """
        
        # Check budget BEFORE making the call
        if self.tracker.today_spend() > self.tracker.daily_budget:
            raise RuntimeError("Daily budget exceeded! Call tracker.report() for details.")
        
        # Route the request
        if model:
            # Specific model requested: put it FIRST in the chain so it is
            # actually used, keeping the others as fallbacks behind it
            fr = self.router.fallback_router
            fr.chain = [model] + [m for m in fr.chain if m != model]
            resp = fr.route(message, system=system, temperature=temperature)
        else:
            # Auto-route to best model for this task
            resp = self.router.route(message, system=system, temperature=temperature)
        
        # Track cost
        self.tracker.log(resp)
        return resp
    
    def report(self):
        self.tracker.report()
        self.router.fallback_router.health_report()

# ── Usage in your app ──
ai = ProductionLLM(daily_budget=5.0)

# Auto-route (default - best model picked automatically)
resp = ai.ask("Write a Python function to calculate compound interest.")
print(f"Answer: {resp.text[:100]}...")

# Specific model (with automatic fallback if it fails)
resp = ai.ask("What is the meaning of life?", model="gemini-2.0-flash")
print(f"Answer: {resp.text[:100]}...")

# With system prompt
resp = ai.ask(
    "Explain transformers to a 10-year-old.",
    system="You are a friendly teacher who uses Indian cricket analogies."
)
print(f"Answer: {resp.text[:100]}...")

# Get the full dashboard
ai.report()

print("""
What ProductionLLM gives you:
  ✅ One .ask() method for everything
  ✅ Auto-routing to the best model per task type
  ✅ Automatic fallback if any provider goes down
  ✅ Retry with exponential backoff
  ✅ Latency monitoring (skips slow providers)
  ✅ Real-time cost tracking with budget alerts
  ✅ Health dashboard showing reliability per provider
  
This is what production AI applications use internally.
""")

# Output: (representative run)
#   Task: code         → Model: gpt-4o                    ($0.00287)
# Answer: def compound_interest(principal: float, rate: float, years: int)...
#   [requested model gemini-2.0-flash moved to front of chain]
# Answer: The meaning of life is a question philosophy has debated for mill...
# Answer: Imagine your cricket team has a coach who watched every match eve...
# (then tracker.report() and health_report() print the dashboards from Part 5)

- `__init__` - assembles the whole hospital: the universal adapter (`UnifiedLLM`), the triage desk (`SmartRouter` which carries Step 3's failover inside it), and the accounts office (`CostTracker`).

- the budget guard at the top of `ask()` - refuses to admit new patients once the daily budget is spent. It runs BEFORE the call, because the only cheaper request than a routed one is the one you never send.

- the `if model:` branch - honors an explicit model request by moving it to the FRONT of the chain (the rest stay behind it as fallbacks). Without that reordering, the parameter would be silently ignored - the bug the original version of this lesson shipped.

- `tracker.log(resp)` - every path out of `ask()` files a receipt; if it isn't logged, it didn't happen (as far as your budget knows).

Read it as: *guard the wallet, route the request, file the receipt - one method, three responsibilities.*

#### 💡 What just happened

What just happened?
  We combined everything into one `ProductionLLM` class with a single `.ask()` method. No model specified → auto-routes to the best model for the task. Model specified → uses it with automatic fallback. Budget exceeded → blocks the request before it costs money. Every call is tracked with cost, latency, and provider reliability. The `.report()` method shows a full dashboard. **This is the foundation of every production AI application.**

---

## 🎯 Concept 7: 2026 Provider Landscape - Groq, Mistral, DeepSeek & More

### 2026 Provider Landscape - Groq, Mistral, DeepSeek & More

8+ providers, each with a niche. Groq for speed. DeepSeek for cost. Mistral for Europe.

| Provider | Flagship Model | Input $/M | Output $/M | Context | Niche |
|---|---|---|---|---|---|
| OpenAI | GPT-4o | $2.50 | $10.00 | 128K | General-purpose leader |
| Anthropic | Claude Sonnet 4.6 | $3.00 | $15.00 | 200K | Best coding (80.8% SWE) |
| Google | Gemini 2.5 Pro | $1.25 | $10.00 | 1M | Strong long-context + price |
| Groq | Llama 3.3 70B | $0.59 | $0.79 | 128K | Fastest (275-1000 tok/s) |
| Mistral | Mistral Large 3 | $0.50 | $1.50 | 256K | EU sovereignty + GDPR |
| DeepSeek | V3 (chat) | $0.27 | $1.10 | 128K | 10-30× cheaper than frontier |
| Fireworks | Llama 3.3 70B | $0.90 | $0.90 | 128K | Fine-tuned at base price |
| Together AI | 200+ models | Varies | Varies | Varies | Broadest open-source catalog |
| Cohere | Command R+ | $2.50 | $10.00 | 128K | Best Rerank API ($2/1K) |
| AWS Bedrock | Multi-model | Varies | Varies | Varies | Enterprise + India region |

#### 💡 What just happened

What just happened?The LLM market fragmented into specialized niches. **No single provider wins everything.** Groq's custom LPU ASICs deliver 2-7× faster inference. DeepSeek V3 costs 10-30× less than OpenAI/Anthropic for comparable tasks. Mistral dropped pricing 8× (from $2/$6 to $0.50/$1.50) while adding 256K context. Together AI offers 200+ open-source models with $100 free credits. Cohere's Rerank API is unique - no other provider offers dedicated semantic reranking. **The 2025-2026 pricing trend:** massive deflation - GPT-4o mini at $0.15/$0.60 is 60× cheaper than GPT-4 Turbo was.

---

## 🎯 Concept 8: Gateway Tools - LiteLLM, OpenRouter, Portkey

### Gateway Tools - LiteLLM, OpenRouter, Portkey

LiteLLM (28K stars, 100+ providers). OpenRouter (300+ models). Portkey (guardrails + caching).

| Tool | Type | Providers | Pricing | Best For |
|---|---|---|---|---|
| LiteLLM | Self-hosted proxy | 100+ | Free (open-source) | Platform teams, data sovereignty |
| OpenRouter | Managed SaaS | 300+ models | 5.5% on credits | Rapid prototyping, experimentation |
| Portkey | Control plane | 1600+ LLMs | Free → $49/mo | Production governance, guardrails |

> ✅ **Info**
>
> 🛠 LiteLLM - one interface, every provider
> `from litellm import completion`
> Same `completion()` call for every provider - just change the model string:
> - `model="gpt-4o"` → OpenAI
> - `model="anthropic/claude-sonnet-4-6"` → Anthropic
> - `model="groq/llama-3.3-70b"` → Groq
> - `model="deepseek/deepseek-chat"` → DeepSeek
> Router and fallback config: [docs.litellm.ai/docs/routing](https://docs.litellm.ai/docs/routing). Proxy server: Docker deploy → any OpenAI SDK client connects. Handles load balancing, fallbacks, virtual keys, spend tracking. 1,500+ req/s at ~156ms latency.

#### 💡 What just happened

What just happened?Gateway tools have **made custom unified wrappers obsolete**. LiteLLM normalizes 100+ providers to OpenAI's interface - self-hosted, free, production-grade. OpenRouter gives instant access to 300+ models via one API key with dynamic routing suffixes (`:nitro` for speed, `:floor` for cheapest). Portkey adds 60+ guardrails (PII blocking, jailbreak detection), semantic caching, and deep observability. **Decision framework:** LiteLLM for self-hosted platform teams. OpenRouter for rapid prototyping. Portkey for production governance. We'll cover streaming responses in Lesson 7.2 and cross-provider function calling in Lesson 7.5; your router's health stats become real dashboards in Lesson 7.6 (Langfuse).

---

## 🎯 Concept 9: Production Patterns - Async, Batch, Structured Output, Error Handling

### Production Patterns - Async, Batch, Structured Output, Error Handling

asyncio.gather for concurrency. Batch API for 50% off. Circuit breakers for resilience.

> 📦 **Info**
>
> ⚙️ Essential production patterns
> - **Async concurrent calls:** `asyncio.gather()` + `Semaphore(10)` for rate limiting. Both OpenAI and Anthropic provide `AsyncOpenAI()` / `AsyncAnthropic()` clients.
> - **Batch API:** Upload JSONL → 50% discount → results within 24hrs. Anthropic batch + cache = 95% reduction ($3.00 → $0.15/M). Use for evaluations, data generation, analytics.
> - **Structured output:** OpenAI: `response_format={"type":"json_schema"}` (44% fewer tokens). Anthropic: via tool_use (11-45% MORE tokens). Google: `response_mime_type="application/json"` (56-61% fewer tokens). Use **Instructor** library for cross-provider Pydantic models.
> - **Error handling stack:** Hard 30s timeout → exponential backoff (2s→4s→8s→30s cap) → circuit breaker (trip after 5 failures, 60s cooldown) → multi-provider fallback → alert at >5% error rate.
> - **Rate limiting:** Token bucket algorithm for dual RPM+TPM limits. Separate limiters per provider/model. Redis for distributed coordination.

#### 💡 What just happened

What just happened?Production LLM systems need **four layers of resilience**: timeouts (hard 30s cap on every call), retries (exponential backoff for transient errors), circuit breakers (prevent cascading failures when a provider goes down), and multi-provider fallbacks (switch providers transparently). **Structured output** varies dramatically: OpenAI's native JSON mode saves 44% tokens, but Anthropic's tool_use approach costs 11-45% MORE tokens - use the Instructor library to abstract these differences. **Batch API** is the single biggest cost lever for non-real-time workloads: 50% off from both OpenAI and Anthropic.

---

## 🎯 Concept 10: India Deployment - DPDPA, Data Residency, Indian Providers

### India Deployment - DPDPA, Data Residency, Indian Providers

No blanket localization required. Bedrock Mumbai works. Sarvam AI for Indic languages.

| Provider | India Region | Models Available | INR Billing |
|---|---|---|---|
| AWS Bedrock | Mumbai + Hyderabad | Claude, Llama, Mistral, DeepSeek, Nova | Yes (GST) |
| Azure OpenAI | Central India | GPT-4o, GPT-4, GPT-3.5 | Yes (GST) |
| Google Vertex AI | Mumbai (asia-south1) | Gemini 2.5 Flash | Yes (GST) |
| OpenAI Direct | ❌ US only | All models | ❌ USD only |
| Anthropic Direct | ❌ US only | All models | ❌ USD only |
| Sarvam AI | India-native | Sarvam-30B/105B, Saaras, Bulbul | ₹ native |
| Krutrim (Ola) | India-native | Krutrim Pro 150B | ₹ native |

> 💡 **Info**
>
> 🇮🇳 DPDPA compliance for LLM APIs
> - **No blanket data localization:** DPDPA uses a blacklist approach. US is NOT blacklisted. Cross-border transfers currently permitted with proper consent + DPAs.
> - **Significant Data Fiduciaries (SDFs):** Face additional localization requirements. Sectors: banking (RBI), telecom, insurance have extra rules.
> - **Practical strategy:** Anonymize PII before API calls. Use India-region endpoints (Bedrock Mumbai) for sensitive data. Maintain Data Processing Agreements. Prepare for potential SDF designation.
> - **Penalties:** Up to ₹250 Cr for non-compliance. Full compliance deadline: May 2027.
> - **Latency from India:** US endpoints add 200-250ms. India-region: 5-20ms. Singapore: 50-80ms.

#### 💡 What just happened

What just happened?India deployment requires careful planning but is tractable. **DPDPA does NOT require blanket data localization** - the US isn't blacklisted, so cross-border API calls are permitted with proper consent. For sensitive data: use **Bedrock Mumbai** (Claude + Llama + DeepSeek with India region), Azure Central India (GPT-4o), or Vertex Mumbai (Gemini Flash). For Indic languages: **Sarvam AI** (Saaras beats GPT-4o on Indian ASR, Sarvam-105B for Hindi/Tamil/Telugu) with ₹ billing and GST compliance. **Krutrim** (Ola) offers 22-language understanding. INR billing via AWS/Azure/GCP India accounts avoids USD conversion hassles.

## Recap

> ℹ️ **Info**
>
> What we built
> - **Three SDKs** - Gemini, OpenAI, Anthropic side by side. Same concept, different syntax.
> - **UnifiedLLM** - one `.chat()` method, any provider. Normalized response with text + tokens + latency + cost.
> - **FallbackRouter** - ordered chain with retry + exponential backoff + latency threshold. Zero downtime.
> - **SmartRouter** - classifies tasks (code/reasoning/creative/structured/simple) and routes to the optimal provider for each.
> - **CostTracker** - per-provider spend, daily budgets, automatic alerts at 80% of limit.
> - **ProductionLLM** - everything combined. One `.ask()` method. Auto-route + fallback + cost control + health monitoring.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Multi-Provider API Integration - Never Depend on One AI**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-7_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-7.1.html` - regenerate this notebook whenever the source HTML is updated.*
